#Incremental Data loading using Autoloader

Schema created 'net_schema' under the catalog called netflix_catalog

In [0]:
%sql
CREATE SCHEMA netflix_catalog.net_schema;

In [0]:
# %sql
# DROP SCHEMA IF EXISTS netflix_catalog.net_schema CASCADE;

Silver is the destination, Can also be in bronze

In [0]:
checkpoint_location = "abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/checkpoint"

In [0]:
# # 1. Delete the autoloader state tracking folder
# dbutils.fs.rm(checkpoint_location, True)

# # 2. Delete the target data folder to prevent duplicates
# dbutils.fs.rm("abfss://bronze@netflixprojectdlsubhik.dfs.core.windows.net/Netflix titles", True)

  readStream - spark streamming API available in PySpark
  Just like format as csv, here mention it as cloudfiles followed by csv using option.

  CloufFiles format is a file format for auto loader
  

In [0]:
df = spark.readStream\
    .format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation",checkpoint_location)\
    .load("abfss://raw@netflixprojectdlsubhik.dfs.core.windows.net")

Output explaination: Databricks simply printing the inferred schema (the structure) of your streaming DataFrame. Newer versions of Databricks (using Spark Connect) automatically list out the DataFrame object type (pyspark.sql.connect.dataframe.DataFrame) and the columns it detected from your CSV file.

it automatically added the _rescued_data:string column
this is a special column generated by Autoloader to safely catch any new or unmapped columns that might arrive in future files (Schema Evolution).

didn't see active "Spark Jobs" running yet In PySpark, reading a stream is a "lazy" operation. This means Spark has looked at the folder, figured out the schema, and prepared the query, but it has not actually started pulling the data yet.

to trigger the Stream To actually initiate the stream and see the active Spark jobs pulling your data, you need to call an action.

In [0]:
# Write the streaming data directly to Bronze
df.writeStream \
    .option("checkpointLocation", checkpoint_location) \
    .trigger(processingTime="10 seconds") \
    .start("abfss://bronze@netflixprojectdlsubhik.dfs.core.windows.net/netflix titles")

as we didn’t mention the data format it wrote in delta format in adls(netflixprojectdlsubhik)> container(bronze)>netflix_titles> part-00000-e69243f8-e665-4686-8336-f2e0922bddd3-c000.snappy.parquet

If new file comes it creates another file 1 with 0 schema and new column it adds in end of files. Autoloader handles this. Creating an external Deltalake/table Not the managed.